In [1]:
!pip install torch transformers accelerate openai spacy
!python -m spacy download en_core_web_trf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 155.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 857.0/857.0 kB 64.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 203.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 128.2 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 160.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.5/35.5 MB 156.5 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 926.3/926.3 kB 62.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 112.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.3/4.3 MB 151.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.3/14.3 MB 186.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 809.2/809.2 kB 69.6 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 28/28 [transformers] [transformers]ub]
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 457.4/457.4 MB 78.7 MB/

In [2]:
import os
from getpass import getpass

if "HF_TOKEN" not in os.environ:
    os.environ["HF_TOKEN"] = getpass("Hugging Face token: ")
if "OPENAI_API_KEY" not in os.environ:
    os.environ["OPENAI_API_KEY"] = getpass("OpenAI API key: ")

Hugging Face token:  ········
OpenAI API key:  ········


In [3]:
from huggingface_hub import login
login(token=os.environ["HF_TOKEN"])

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.


In [4]:
import openai
openai.api_key = os.environ["OPENAI_API_KEY"]

In [5]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B", add_prefix_space=True)
model = AutoModelForCausalLM.from_pretrained(
    "meta-llama/Llama-3.1-8B",
    dtype=torch.float16,
    device_map="auto"
)
device = next(model.parameters()).device
device

config.json:   0%|          | 0.00/826 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/50.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

device(type='cuda', index=0)

In [6]:
import spacy
import string
import traceback
import random
import re
from typing import Optional
nlp = spacy.load("en_core_web_trf")

In [7]:
def is_punctuation(token):
    return all(char in ".’,!?;:-()[]{}\"'" for char in token)

def is_valid_english_token(token):
    if any(char in token for char in [":", "\\"]):
        return False
    if token == "...":
        return False
    return bool(re.fullmatch(r"[a-zA-Z0-9'\".,!?;’%$€£¥]*", token))

aux_token = ["aren", "wer", "weren", "wasn", "doesn", "didn", "couldn", "shan", "cannot", "didnt", "dont", "don", "youre", "im", "theyre"]

def in_aux_token(token):
    return token.lower() in aux_token

def is_new_word(token_str: str) -> bool:
    return token_str.startswith("Ġ") or token_str.startswith("▁") or token_str.startswith(" ")

def clean_token(token_str: str) -> str:
    return token_str.strip().lstrip("Ġ▁ ")

def get_contextual_pos(nlp, context: str, candidate: str) -> Optional[str]:
    if is_punctuation(clean_token(candidate)):
        combined_text = (context + candidate).strip()
    else:
        combined_text = (context + " " + candidate).strip()
    doc = nlp(combined_text)
    if doc:
        return doc[-1].pos_
    return None

def get_contextual_pos_append(nlp, context: str, candidate: str) -> Optional[str]:
    combined_text = (context + candidate).strip()
    doc = nlp(combined_text)
    if doc:
        return doc[-1].pos_
    return None

def get_generated_tokens(nlp, prompt: str, full_text: str):
    doc = nlp(full_text)
    return [(token.text, token.pos_) for token in doc if token.idx >= len(prompt)]

In [8]:
def has_single_pos(combined_word):
    combined_pos = nlp(combined_word)
    pos_tags = set(token.pos_ for token in combined_pos if token.pos_ != "SPACE")
    return len(pos_tags) == 1

def check_pos_tags(nlp, context: str, candidate: str) -> set[str]:
    combined_text = (context + candidate).strip()
    doc = nlp(combined_text)
    if not doc:
        return set()
    candidate_pos_tags = set()
    last_token = doc[-1]
    if last_token.pos_ != "SPACE":
        candidate_pos_tags.add(last_token.pos_)
    return candidate_pos_tags

def starts_with_uppercase(text):
    text = text.lstrip()
    return text[0].isupper() if text else False

def is_all_letters(s):
    return re.fullmatch(r"[A-Za-z]+", s) is not None

def is_all_punct(s):
    return re.fullmatch(r"[^\w\s]+", s) is not None

In [9]:
def generate_allowed_pos_transitions(encode_bit_1: list[str], encode_bit_0: list[str], neutral_pos: list[str]) -> set[tuple[str, str]]:
    
    allowed = set()

    def add_symmetric(pos_list):
        for p1 in pos_list:
            for p2 in pos_list:
                if p1 != p2:
                    allowed.add((p1, p2))

    add_symmetric(encode_bit_1)
    add_symmetric(encode_bit_0)
    add_symmetric(neutral_pos)

    for group in [encode_bit_1, encode_bit_0, neutral_pos]:
        for p in group:
            allowed.add((p, p))
    
    for group in (encode_bit_1, encode_bit_0, neutral_pos):
        if "PROPN" in group and "ADJ" in group:
            allowed.discard(("PROPN", "ADJ"))
            allowed.discard(("ADJ", "PROPN"))
            allowed.discard(("NOUN", "PROPN"))
            allowed.discard(("PROPN", "NOUN"))

    return allowed

In [10]:
def is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions, verbose=True) -> bool:
    disallowed_change_found = False
    for old_token, new_token in zip(stable_tokens, new_stable_tokens):
        if old_token[1] != new_token[1] and (old_token[1], new_token[1]) not in allowed_transitions:
            if verbose:
                print(f"    -> Skipped candidate (POS change for token '{old_token[0]}': {old_token[1]} → {new_token[1]})")
            disallowed_change_found = True
    return disallowed_change_found

def is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) -> bool:
    return is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions, verbose=False)

In [11]:
def simulate_next_bit_feasibility(model, tokenizer, nlp, context: str, next_bit: str, prompt: str, last_token_id: int, 
                                  candidate_id: int, encode_bit_1: list[str], encode_bit_0: list[str], neutral_pos: list[str], 
                                  allowed_transitions: set[tuple[str,str]], top_k: int = 20) -> bool: 
    inputs = tokenizer(context, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]
    outputs = model(**inputs)
    logits = outputs.logits[0, -1]
    topk_values, topk_indices = torch.topk(logits, top_k)
    found_pos_match = False
    found_neutral = False

    for i, token_id in enumerate(topk_indices):
        candidate = tokenizer.decode([token_id])
        cleaned = clean_token(candidate)
        if is_new_word(candidate) or is_punctuation(cleaned):
            pos = get_contextual_pos(nlp, context, cleaned)
        else:
            pos = get_contextual_pos_append(nlp, context, cleaned)
        if is_new_word(candidate):
            current_full_text = (context + " " + cleaned).strip()
        else:
            current_full_text = (context + cleaned).strip()
            
        candidate_tensor = torch.tensor([[token_id.item()]], device=device)
        simulated_input_ids = torch.cat([input_ids, candidate_tensor], dim=1)
        stable_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        simulated_full_text = tokenizer.decode(simulated_input_ids[0], skip_special_tokens=True)
        stable_tokens = get_generated_tokens(nlp, prompt, stable_text)
        new_stable_tokens = get_generated_tokens(nlp, prompt, simulated_full_text)
        
        if pos is None:
            print("    [SKIP] POS is None.")
            continue
        if (pos in neutral_pos) and not is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) and (candidate_id != token_id) and is_valid_english_token(cleaned):
            found_neutral = True
        if ((next_bit == "1" and pos in encode_bit_1) or 
            (next_bit == "0" and pos in encode_bit_0)) and \
            not is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions) and \
            is_valid_english_token(cleaned) and (candidate_id != token_id):
            found_pos_match = True

    if not found_pos_match and not found_neutral:
        print(f"    -> BOTH Feasibility check failed: BOTH no neutral nor suitable POS in top-{top_k}.")
    return found_pos_match or found_neutral

In [12]:
def encode_hidden_message(model, tokenizer, nlp, prompt: str, hidden_message: str, encode_bit_1, encode_bit_0, neutral_pos, feas = True) -> tuple[str, bool]:
    torch.cuda.empty_cache()
    input_ids = tokenizer(prompt, return_tensors="pt").input_ids.to(device)
    print(f"Initial input tokens: {tokenizer.convert_ids_to_tokens(input_ids[0])}")
    allowed_transitions = generate_allowed_pos_transitions(encode_bit_1, encode_bit_0, neutral_pos)
    exceeded_topk = False
    hidden_token_ids = []
    current_bit_index = 0
    carry_bit: Optional[str] = None
    last_word = ""
    last_word_pos: Optional[str] = None
    last_token_id = input_ids[0][-1].item() 
    ner_counter = 0
    base_top_k = 5
    max_top_k = 1280
    prev_top_k = 0
    outputs = model(input_ids=input_ids)
    next_token_logits = outputs.logits[0, -1, :].to(device)

    while current_bit_index < len(hidden_message) or carry_bit is not None:
        bit_to_encode = carry_bit if carry_bit is not None else hidden_message[current_bit_index]
        bit_info = f"carry '{bit_to_encode}'" if carry_bit is not None else f"'{bit_to_encode}' (index {current_bit_index})"
        print(f"\nEncoding bit {bit_info}...")
        candidate_found = False
        selected_token_id = None
        current_top_k = base_top_k
        current_full_text = tokenizer.decode(input_ids[0], skip_special_tokens=True)
        stable_tokens = get_generated_tokens(nlp, prompt, current_full_text)

        while not candidate_found and current_top_k <= max_top_k:
            ideal_candidate = None
            acceptable_candidate = None
            higher_six = False
            highest_candidate = None
            highest_candidate_id = None
            highest_candidate_pos = None
            highest_prob = None
            highest_not = None
            topk_values, topk_indices = torch.topk(next_token_logits, current_top_k)
            print(f"Trying candidates {prev_top_k+1}–{current_top_k} of top-{current_top_k}:")
            
            for j, candidate_id in enumerate(topk_indices[prev_top_k:current_top_k], start=prev_top_k):
                candidate_str = tokenizer.decode(candidate_id.item())
                raw_candidate_str = tokenizer.convert_ids_to_tokens(candidate_id.item())
                candidate_prob = topk_values[j].item()
                candidate_tensor = torch.tensor([[candidate_id.item()]], device=device)
                simulated_input_ids = torch.cat([input_ids, candidate_tensor], dim=1)
                simulated_full_text = tokenizer.decode(simulated_input_ids[0], skip_special_tokens=True)
                new_stable_tokens = get_generated_tokens(nlp, prompt, simulated_full_text)
                print(f"  Candidate {j+1}: token='{candidate_str}', raw='{raw_candidate_str}', probability='{candidate_prob}', ID={candidate_id}", end="")
                
                if j == 0:
                    highest_prob = candidate_prob
                    highest_candidate = candidate_str
                    highest_candidate_id = candidate_id.item()

                if re.search(r"Ċ", raw_candidate_str) or re.search(r"<|end_of_text|>", raw_candidate_str):
                    print("    -> Skipped (invalid characters in token)")
                    if j==0:
                        highest_not = True
                    continue
                
                if last_token_id == candidate_id:
                    print("Skip! The same candidate two consecutive times!")
                    continue
                
                if not candidate_str.strip():
                    print("    -> Skipped (empty/whitespace)")
                    if j==0:
                        highest_not = True
                    continue
                
                if last_word_pos == "X" and is_new_word(candidate_str):
                    print("Skip! X POS must be appended, not a new word!")
                    if j==0:
                        highest_not = True
                    continue
                    
                if candidate_id == 20182:
                    print("Invalid token - skip!")
                    continue
                    
                clean_candidate = clean_token(candidate_str)
                if not is_valid_english_token(clean_candidate):
                    print("    -> Skipped (invalid characters in token)")
                    if j==0:
                        highest_not = True
                    continue
                     
                allowed_exceptions = {"'s", "’s", "'t", "’t"}
                punctuation_count = sum(1 for c in clean_candidate if c in string.punctuation)
                if (clean_candidate not in allowed_exceptions and (punctuation_count > 1 or (punctuation_count == 1 and len(clean_candidate.strip(string.punctuation)) > 0))):
                    print("    -> Skipped (punctuation + character or more than one punctuation)")
                    if j==0:
                        highest_not = True
                    continue
                    
                if last_word_pos == "PUNCT" and not is_new_word(candidate_str):
                    print("Skip! After comma, there must be a new word")
                    if j==0:
                        highest_not = True
                    continue
                    
                full_context = tokenizer.decode(input_ids[0], skip_special_tokens=True)
                if is_new_word(candidate_str) or is_punctuation(clean_candidate):
                    pos = get_contextual_pos(nlp, full_context, clean_candidate)
                          
                    if is_all_letters(clean_candidate) and pos == "PUNCT":
                        print(f"Skipping - Classified as PUNCT but contains only letters.")
                        continue

                    if is_all_punct(clean_candidate) and pos != "PUNCT":
                        print(f"Skipping - Classified as {pos} but contains only punctuation.")
                        continue
                    
                    if in_aux_token(clean_candidate):
                        pos = "VERB"
                        print(f"POS Special AUX Case! = {pos}", end="\n")
                    else:
                        print(f"POS = {pos}", end="\n")
                    
                    if ner_counter <= 2 and last_word_pos == "PROPN" and pos == "PROPN" and starts_with_uppercase(last_word) and starts_with_uppercase(clean_candidate) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        print("Enter NER condition")
                        
                        if acceptable_candidate is None:
                            print(f"  → Recording NER-style acceptable candidate: '{candidate_str}'")
                            acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                        carry_bit = bit_to_encode
                        if j == 0:
                            print("  NER j == 0 – breaking to use as fallback")
                            break
                        else:
                            print("  NER j > 0 – will keep looking for ideal")
                            continue
                    elif ner_counter > 2 and last_word_pos == "PROPN" and pos == "PROPN" and starts_with_uppercase(last_word) and starts_with_uppercase(clean_candidate) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        print("NER Counter overflow!")
                        continue
                        
                    if pos == "X":
                        print("Skip X token!")
                        continue
                        
                    if candidate_id.item() == 364:
                        print("Skip id 364 token!")
                        continue
                        
                    if current_bit_index + 1 < len(hidden_message): 
                        next_bit = hidden_message[current_bit_index + 1]
                    else:
                        next_bit = None
                    
                    if pos is None:
                        print("    -> Skipped (could not determine POS for appended token)")
                        if j==0:
                            highest_not = True
                            continue
                
                    if j == 0:
                        highest_candidate_pos = pos
                    
                    if bit_to_encode == "0":
                        if (pos in encode_bit_0):
                            candidate_quality = "ideal"
                        elif pos in encode_bit_1:
                            print("    -> Skipped (POS BIT 1 not allowed for bit 0)")
                            if j==0:
                                highest_not = True
                            continue
                        else:
                            candidate_quality = "acceptable"
                    else:
                        if pos in encode_bit_1:
                            candidate_quality = "ideal"
                        elif (pos in encode_bit_0):
                            print("    -> Skipped (POS BIT 0 not allowed for bit 1)")
                            if j==0:
                                highest_not = True
                            continue    
                        else:
                            candidate_quality = "acceptable"

                    if len(new_stable_tokens) < len(stable_tokens):
                        print("    -> Skipped candidate (fewer stable tokens)")
                        if j==0:
                            highest_not = True
                        continue

                    if is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        if j==0:
                            highest_not = True
                        continue
                    
                    if is_punctuation(clean_token(clean_candidate)):
                        text_to_check = (full_context + clean_candidate).strip()
                    else:
                        text_to_check = (full_context + " " + clean_candidate).strip()

                    if acceptable_candidate is None or candidate_quality == "ideal":
                        if feas:
                            if next_bit:
                                feasible = simulate_next_bit_feasibility(model, tokenizer, nlp, text_to_check, next_bit, prompt, last_token_id, candidate_id, 
                                                                         encode_bit_1, encode_bit_0, neutral_pos, allowed_transitions, 20)
                                if not feasible:
                                    print("Skip! No feasible token next")
                                    continue          
                    
                    if candidate_quality == "ideal":
                        ideal_candidate = (candidate_id.item(), candidate_str, pos)
                        break
                    elif acceptable_candidate is None:
                        acceptable_candidate = (candidate_id.item(), candidate_str, pos)

                else:
                    
                    if last_word == "":
                        print("    -> Skipped (no previous word to append to)")
                        if j==0:
                            highest_not = True
                        continue
                        
                    if current_bit_index == 0:
                        print("    -> Skipped (First token after the prompt must be a new word!)")
                        if j==0:
                            highest_not = True
                        continue

                    pos = get_contextual_pos_append(nlp, full_context, candidate_str)

                    if pos is None:
                        print("    -> Skipped (could not determine POS for appended token)")
                        if j==0:
                            highest_not = True
                        continue
                        
                    combined_word = last_word + candidate_str
                    appending = nlp(candidate_str)
                    pos_appending = appending[0].pos_
                    if in_aux_token(clean_candidate):
                        pos_appending = "VERB"
                        print(f"POS Special AUX Case! = {pos_appending}")
                    print(f", POS = {pos_appending}")
                    check_pos_tags(nlp, full_context, candidate_str)

                    if (has_single_pos(combined_word) and (((last_word_pos in encode_bit_0) and (pos in encode_bit_0)) or ((last_word_pos in encode_bit_1) and (pos in encode_bit_1))) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions)):
                        
                        if ((j == 0)):
                            print(f"1.Ideal/Acceptable candidate token: '{candidate_str}' to form '{combined_word}'. POS of combined word: '{pos}' --> j == 0 or 1 case")
                            candidate_quality = "acceptable"
                            acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                            carry_bit = bit_to_encode
                            candidate_found = False
                            break
                        else:
                            candidate_quality = "acceptable"
                            print(f"1.5Acceptable candidate token: '{candidate_str}' to form '{combined_word}', POS of combined word = {pos}")
                    elif has_single_pos(combined_word) and carry_bit is not None and bit_to_encode == "1" and (last_word_pos in neutral_pos) and pos in encode_bit_1 and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "ideal"
                        print(f"2Ideal appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif has_single_pos(combined_word) and carry_bit is not None and bit_to_encode == "0" and (last_word_pos in neutral_pos) and pos in encode_bit_0 and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "ideal"
                        print(f"3Ideal appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif has_single_pos(combined_word) and (last_word_pos in neutral_pos) and (pos in neutral_pos) and not is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        candidate_quality = "acceptable"
                        print(f"3.5Acceptable appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    elif (candidate_id.item() == 753 or candidate_id.item() == 596 or candidate_id.item() == 956 or candidate_id.item() == 1431):
                        candidate_quality = "acceptable"
                        print(f"4.5Acceptable appending candidate token: '{candidate_str}' to form '{combined_word}'")
                    else:
                        print(f"   -> Skipped: wrong appending POS, pos_appending = {pos_appending} for candidate '{candidate_str}', combined contextual POS: {pos}")
                        if is_pos_shift_detected_noprint(stable_tokens, new_stable_tokens, allowed_transitions):
                            print(f"    -> Skipped: POS change detected!")
                        if not has_single_pos(combined_word):
                            print(f"    -> Skipped: Combined POS longer than 1!")
                            if j==0:
                                highest_not = True
                        continue

                    if len(new_stable_tokens) < len(stable_tokens):
                        print("    -> Skipped candidate (fewer stable tokens)")
                        if j==0:
                            highest_not = True
                        continue

                    if is_pos_shift_detected(stable_tokens, new_stable_tokens, allowed_transitions):
                        if j==0:
                            highest_not = True
                        continue
                    
                    if candidate_quality == "ideal":
                        ideal_candidate = (candidate_id.item(), candidate_str, pos)
                        break

                    elif acceptable_candidate is None:
                        acceptable_candidate = (candidate_id.item(), candidate_str, pos)
                        
                
            if (
                highest_candidate is not None
                and highest_not is not True
                and highest_candidate_pos != "PUNCT"
                and not is_punctuation(clean_token(highest_candidate))
                and is_new_word(candidate_str)
                and ideal_candidate is not None
                and highest_prob - candidate_prob > 6.1
                and (current_top_k < 11)
                and highest_candidate_pos in neutral_pos
            ):
                print(f"    -> Selected highest probability candidate '{highest_candidate}' (ID {highest_candidate_id}) as acceptable due to probability margin > 6.10")
                acceptable_candidate = (highest_candidate_id, highest_candidate, highest_candidate_pos)
                higher_six = True
                candidate_found = False
            
            if ideal_candidate is not None and not higher_six:
                selected_token_id, token_str, token_pos = ideal_candidate
                if is_new_word(token_str) or is_punctuation(token_str):
                    last_word = token_str
                else:
                    last_word += token_str
                last_word_pos = token_pos
                last_token_id = selected_token_id
                candidate_found = True

                if carry_bit is not None:
                    carry_bit = None
                current_bit_index += 1 
            
                if last_word_pos != "PROPN":
                    ner_counter = 0
                elif starts_with_uppercase(token_str) and token_pos == "PROPN":
                    ner_counter = ner_counter + 1
                    print(f"NER Counter: {ner_counter}, ideal case")
                print(f"    --> Selected new word token (IDEAL): '{token_str}' for bit {bit_to_encode}")
                
                
            elif acceptable_candidate is not None and not candidate_found:
                selected_token_id, token_str, token_pos = acceptable_candidate
                if is_new_word(token_str) or is_punctuation(token_str):
                    last_word = token_str
                else:
                    last_word += token_str
                last_token_id = selected_token_id
                last_word_pos = token_pos
                candidate_found = True
                carry_bit = bit_to_encode
                higher_six = False
                if last_word_pos != "PROPN":
                    ner_counter = 0
                elif starts_with_uppercase(token_str) and token_pos == "PROPN":
                    ner_counter = ner_counter + 1
                    print(f"NER Counter: {ner_counter}, acceptable case")
                print(f"    --> Selected new word token (ACCEPTABLE): '{token_str}' for bit {bit_to_encode} (carrying)")
            
            if not candidate_found:
                print(f"Expanding candidate pool from top-{current_top_k}...")
                prev_top_k = current_top_k
                if current_top_k < 30:
                    current_top_k += 5
                elif current_top_k <= 130:
                    current_top_k += 50
                else:
                    current_top_k += 100
            else:
                prev_top_k = 0
                
            if carry_bit is not None:
                current_top_k = int(current_top_k + 5)
                base_top_k = min(current_top_k, max_top_k)
            else:
                base_top_k = 5

            if current_top_k > 530:
                print("Top-k limit reached. Ending generation early.")
                exceeded_topk = True
                break
                
        if selected_token_id is None:
            print("Warning: no candidate found, ending early.")
            return tokenizer.decode(hidden_token_ids, skip_special_tokens=True), True

        hidden_token_ids.append(selected_token_id)
        token_tensor = torch.tensor([[selected_token_id]], device=device)
        input_ids = torch.cat([input_ids, token_tensor], dim=1)
        outputs = model(input_ids=input_ids)
        next_token_logits = outputs.logits[0, -1, :].to(device)

    stable_text       = tokenizer.decode(input_ids[0], skip_special_tokens=True)
    frozen_prefix     = get_generated_tokens(nlp, prompt, stable_text)
    frozen_prefix_len = len(frozen_prefix)

    max_new            = 40
    top_k              = 1000   
    top_p              = 0.9
    temperature        = 0.8
    repetition_penalty = 1.2

    last_id = input_ids[0, -1].item()

    for i in range(max_new):
        outputs = model(input_ids=input_ids)
        logits  = outputs.logits[0, -1, :]
        for prev in set(input_ids.flatten().tolist()):
            logits[prev] /= repetition_penalty
        logits = logits / temperature
        values, indices = torch.topk(logits, top_k)

        picked_id   = None
        picked_clean = None

        for _, idx in zip(values, indices):
            cand_id = idx.item()
            if cand_id == last_id:
                continue
                
            tok   = tokenizer.decode(cand_id)
            clean = clean_token(tok)
            
            if i == 0 and not (is_new_word(tok) or is_punctuation(clean)):
                continue
            cand_tensor   = torch.tensor([[cand_id]], device=input_ids.device)
            sim_ids       = torch.cat([input_ids, cand_tensor], dim=1)
            sim_text      = tokenizer.decode(sim_ids[0], skip_special_tokens=True)
            new_full      = get_generated_tokens(nlp, prompt, sim_text)
            new_prefix    = new_full[:frozen_prefix_len]

            if is_pos_shift_detected_noprint(frozen_prefix, new_prefix, allowed_transitions):
                continue

            picked_id = cand_id
            picked_clean = clean
            break

        if picked_id is None:
            break

        hidden_token_ids.append(picked_id)
        input_ids = torch.cat([input_ids,
                               torch.tensor([[picked_id]], device=input_ids.device)],
                              dim=1)
        last_id = picked_id

        if picked_clean in {".", "?", "!"}:
            break

    generated_text = tokenizer.decode(hidden_token_ids, skip_special_tokens=True)
    return generated_text, exceeded_topk

In [13]:
def decode_hidden_message(prompt, text, hidden_message, nlp,
                          encode_bit_1, encode_bit_0, neutral_pos):
    full_text = prompt + text
    doc = nlp(full_text)
    prompt_len = len(prompt)
    exploded = []
    i = 0
    
    while i < len(doc):
        tok = doc[i]
        
        if tok.text.lower() == "can" and i+1 < len(doc) and doc[i+1].text.lower() == "not":
            exploded.append(("cannot", "VERB", tok.idx))
            i += 2
            continue
        
        if i+1 < len(doc) and doc[i+1].text in {"n't", "n’t"}:
            base = tok.text + "n"
            exploded.append((base, tok.pos_, tok.idx))
            exploded.append(("'t", "PART", doc[i+1].idx))
            i += 2
            continue
            
        if tok.text.lower() in {"you", "they"}:
            if i+1 < len(doc) and doc[i+1].text.lower() in {"re"}:
                merged = tok.text + doc[i+1].text
                exploded.append((merged, "VERB", tok.idx))
                i += 2
                continue
        
        exploded.append((tok.text, tok.pos_, tok.idx))
        i += 1

    decoded = ""
    last_pos = None
    last_text = ""
    print("Decoding tokens:")
    for word, pos, idx in exploded:
        if in_aux_token(word):
            pos = "VERB"
        if idx < prompt_len:
            last_pos, last_text = pos, word
            continue
            
        if len(decoded) == len(hidden_message):
            break

        if last_pos == "PROPN" and pos == "PROPN" \
           and last_text[:1].isupper() and word[:1].isupper():
            print(f"Word: '{word}', POS: {pos} -> Skipped (NER‑style)")
        elif word in {"'s", "’s", "'t", "’t"}:
            print(f"Word: '{word}', POS: {pos} -> Skipped (contraction)")
        elif pos in encode_bit_0:
            decoded += "0"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 0")
        elif pos in encode_bit_1:
            decoded += "1"
            print(f"Word: '{word}', POS: {pos} -> Decoded as 1")
        else:
            print(f"Word: '{word}', POS: {pos} -> Skipped (neutral/other)")
        last_pos, last_text = pos, word

    print(f"\nD: {decoded}")
    print(f"H: {hidden_message}")
    if decoded == hidden_message:
        print("Decoded message matches!")
        return decoded, True
    else:
        print("Mismatch.")
        return decoded, False

In [14]:
def chat_with_gpt(text: str) -> str:
    prompt = (
        f"Rate the following text on a scale from 1 to 100, where 100 means perfect in terms of mistakes, coherence, fluency, grammar, naturalness, and truthfulness, and 1 means it's completely incoherent, full of mistakes or unnatural. As a response just return the number! Text to rate:\n\n\"{text}\"")
    response = openai.chat.completions.create(
        model="gpt-4.1",
        messages=[
            {"role": "system", "content": "You are a helpful and precise text quality evaluator."},
            {"role": "user", "content": prompt}
        ],
        temperature=0.0,
    )
    return response.choices[0].message.content.strip()

In [15]:
encode_bit_1 = ["VERB", "ADP", "SCONJ", "DET", "PRON", "ADJ", "PUNCT"]
encode_bit_0 = ["ADV", "AUX", "PART", "PROPN", "NOUN", "CCONJ", "NUM"]
neutral_pos = ["INTJ", "SYM"]

In [16]:
import json
import pandas as pd

LENGTH_PROMPTS = [
    "Books provide windows into countless worlds and ideas, serving as enduring sources of knowledge, entertainment, and inspiration.",
    "Artificial intelligence is transforming industries by automating tasks, analyzing vast data sets, and enabling more personalized experiences.",
    "Psychology is the scientific study of the mind and behavior, seeking to understand why individuals think, feel, and act the way they do.",
    "Plants need sunlight and water to grow.",
    "Regular exercise is important for maintaining good health.",
    "Travel opens your mind and fills your life with unforgettable memories.",
]

SECRET_SENTENCES = [
    "Warsaw University of Technology produces world-class engineers.",
    "Artificial intelligence is one of the most rapidly evolving technologies.",
    "Meta Llama 3.1 is a multilingual large language model.",
    "LLM inference is the process of a trained model generating a response to a prompt.",
]

BIT_LENGTHS = list(range(16, 112 + 1, 16))
LENGTH_RESULTS_FILE = "../results/length_experiment_results.jsonl"


def text_to_bits(text):
    return "".join(format(b, "08b") for b in text.encode("iso-8859-2"))


SENTENCE_BITS = [text_to_bits(s) for s in SECRET_SENTENCES]

In [17]:
import gc
import time


def run_length_experiment(model, tokenizer, nlp, encode_bit_1, encode_bit_0, neutral_pos,
                          feas=True, out_path=LENGTH_RESULTS_FILE):
    with open(out_path, encoding="utf-8") as f:
        existing = [json.loads(line) for line in f if line.strip()]
    done = {(r["prompt_idx"], r["sentence_idx"], r["n_bits"]) for r in existing}
    failed_combos = {(r["prompt_idx"], r["sentence_idx"]) for r in existing if r["status"] in ("topk_exceeded", "oom")}
    records = []
    with open(out_path, "a", encoding="utf-8") as f:
        for n_bits in BIT_LENGTHS:
            for p_idx, prompt in enumerate(LENGTH_PROMPTS):
                for s_idx, sent_bits in enumerate(SENTENCE_BITS):
                    if (p_idx, s_idx, n_bits) in done or (p_idx, s_idx) in failed_combos:
                        continue

                    msg = sent_bits[:n_bits]
                    t0 = time.time()
                    status, rating, match, text = "ok", None, None, None

                    try:
                        text, exceeded = encode_hidden_message(
                            model, tokenizer, nlp, prompt, msg,
                            encode_bit_1, encode_bit_0, neutral_pos, feas=feas)
                        if exceeded:
                            status = "topk_exceeded"
                            rating = 1.0
                        else:
                            _, match = decode_hidden_message(
                                prompt, text, msg, nlp,
                                encode_bit_1, encode_bit_0, neutral_pos)
                            rating = float(chat_with_gpt(text))
                    except torch.cuda.OutOfMemoryError:
                        status = "oom"
                        failed_combos.add((p_idx, s_idx))
                        torch.cuda.empty_cache()
                        gc.collect()
                    except Exception as e:
                        status = "error"
                        print(f"[WARN] Error for P{p_idx} S{s_idx}, {n_bits} bits: {e}")

                    if status == "topk_exceeded":
                        failed_combos.add((p_idx, s_idx))

                    rec = {
                        "n_bits": n_bits, "n_chars": n_bits // 8,
                        "prompt_idx": p_idx, "sentence_idx": s_idx,
                        "hidden_message": msg, "status": status,
                        "rating": rating, "decode_match": match,
                        "generated_text": text, "elapsed_s": round(time.time() - t0, 1),
                    }
                    records.append(rec)
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
                    f.flush()
                    print(f"{n_bits} bits, P{p_idx} S{s_idx}: {status}, rating={rating}, match={match}")

    return pd.DataFrame(records)

In [ ]:
df_length = run_length_experiment(model, tokenizer, nlp, encode_bit_1, encode_bit_0, neutral_pos)
print(df_length["status"].value_counts())

Initial input tokens: ['<|begin_of_text|>', 'Books', 'Ġprovide', 'Ġwindows', 'Ġinto', 'Ġcountless', 'Ġworlds', 'Ġand', 'Ġideas', ',', 'Ġserving', 'Ġas', 'Ġenduring', 'Ġsources', 'Ġof', 'Ġknowledge', ',', 'Ġentertainment', ',', 'Ġand', 'Ġinspiration', '.']


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.



Encoding bit '0' (index 0)...
Trying candidates 1–5 of top-5:
  Candidate 1: token=' They', raw='ĠThey', probability='16.421875', ID=2435POS = PRON
    -> Skipped (POS BIT 1 not allowed for bit 0)
  Candidate 2: token=' In', raw='ĠIn', probability='15.9921875', ID=763POS = ADP
    -> Skipped (POS BIT 1 not allowed for bit 0)
  Candidate 3: token=' However', raw='ĠHowever', probability='15.96875', ID=4452POS = ADV
    --> Selected new word token (IDEAL): ' However' for bit 0

Encoding bit '1' (index 1)...
Trying candidates 1–5 of top-5:
  Candidate 1: token=',', raw=',', probability='22.171875', ID=11POS = PUNCT
    --> Selected new word token (IDEAL): ',' for bit 1

Encoding bit '0' (index 2)...
Trying candidates 1–5 of top-5:
  Candidate 1: token=' the', raw='Ġthe', probability='15.578125', ID=279POS = PRON
    -> Skipped (POS BIT 1 not allowed for bit 0)
  Candidate 2: token=' not', raw='Ġnot', probability='14.9140625', ID=539POS = PART
    --> Selected new word token (IDEAL): ' not